In [1]:
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler, ChiSqSelector
from pyspark.ml.stat import Correlation
from pyspark.sql.types import DoubleType

class FeatureSelectorBDA:
    def __init__(self, df, target_col):
        self.df = df
        self.target_col = target_col  

    def label_indexed(self):
        dtype = dict(self.df.dtypes)[self.target_col]
        if dtype == "string":
            temp_col = self.target_col + "_idx"  
            indexer = StringIndexer(inputCol=self.target_col, outputCol=temp_col, handleInvalid="keep")
            self.df = indexer.fit(self.df).transform(self.df)
            self.df = self.df.drop(self.target_col).withColumnRenamed(temp_col, self.target_col)
            self.df = self.df.withColumn(self.target_col, col(self.target_col).cast(DoubleType()))
        else:
            self.df = self.df.withColumn(self.target_col, col(self.target_col).cast(DoubleType()))
        return self.df

    def division_columnas(self):
        cat_cols = [c for (c, t) in self.df.dtypes if t == "string" and c != self.target_col]
        num_cols = [c for (c, t) in self.df.dtypes if t in ("double", "float", "int") and c != self.target_col]
        print(f"Categóricas: {cat_cols}")
        print(f"Numéricas: {num_cols}\n")
        return cat_cols, num_cols

    def get_multicolinealidad(self, threshold=0.9):
        num_cols = [c for (c, t) in self.df.dtypes if t in ("double", "float", "int") and c != self.target_col]
        if not num_cols:
            print("No hay columnas numéricas para evaluar multicolinealidad.")
            return [], [], None

        assembler = VectorAssembler(inputCols=num_cols, outputCol="num_vec")
        df_vec = assembler.transform(self.df).select("num_vec")

        corr_matrix = Correlation.corr(df_vec, "num_vec", "pearson").head()[0].toArray()

        keep_idx, to_drop = [], []
        for i in range(len(corr_matrix)):
            if all(abs(corr_matrix[i][j]) < threshold for j in range(i)):
                keep_idx.append(i)
            else:
                to_drop.append(num_cols[i])

        num_cols_filtered = [num_cols[i] for i in keep_idx]

        print(f"Variables numéricas seleccionadas tras multicolinealidad: {num_cols_filtered}")
        print(f"Variables descartadas: {to_drop}\n")
        return num_cols_filtered, to_drop, corr_matrix

    def get_cols_selected(self, is_categorical, input_cols, threshold=0.1, num_top_features=5, force_include_cols=None):
        if not input_cols:
            print("No hay columnas de entrada para analizar.")
            return []
    
        if force_include_cols is None:
            force_include_cols = []
    
        df_temp = self.df.select(input_cols + [self.target_col])
    
        if is_categorical:
            indexed_cols = []
            for c in input_cols:
                dtype = dict(self.df.dtypes)[c]
                if dtype == "string":
                    temp_col = c + "_idx"
                    indexer = StringIndexer(inputCol=c, outputCol=temp_col, handleInvalid="keep")
                    df_temp = indexer.fit(df_temp).transform(df_temp)
                    indexed_cols.append(temp_col)
                else:
                    indexed_cols.append(c)
    
            assembler = VectorAssembler(inputCols=indexed_cols, outputCol="cat_features")
            df_vec = assembler.transform(df_temp).select("cat_features", self.target_col)
    
            selector = ChiSqSelector(
                featuresCol="cat_features",
                outputCol="selectedFeatures",
                labelCol=self.target_col,
                numTopFeatures=num_top_features
            )
            model = selector.fit(df_vec)
            selected_indices = model.selectedFeatures
            selected_cols = [input_cols[i] for i in selected_indices]
            print(f"Categóricas seleccionadas (Chi²): {selected_cols}\n")
    
        else:
            assembler = VectorAssembler(inputCols=input_cols, outputCol="num_features")
            df_vec = assembler.transform(df_temp).select("num_features", self.target_col)
    
            corr_matrix = Correlation.corr(df_vec, "num_features", "pearson").head()[0].toArray()
            corr_values = [abs(corr_matrix[i][0]) for i in range(len(input_cols))]
            selected_cols = [input_cols[i] for i, c in enumerate(corr_values) if c > threshold]
            for col_name in force_include_cols:
                if col_name in input_cols and col_name not in selected_cols:
                    selected_cols.append(col_name)
    
            print(f"Numéricas seleccionadas (corr > {threshold}): {selected_cols}\n")
    
        return selected_cols